In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [4]:
import pandas as pd
query = pd.read_csv('queries.csv')['query'][:50]

In [5]:
query

,query
0,Generate a podcast content between two people ...
1,Generate a podcast content between two people ...
2,Generate a podcast content between two people ...
3,Generate a podcast content between two people ...
4,Generate a podcast content between two people ...
5,Generate a podcast content between two people ...
6,Generate a podcast content between two people ...
7,Generate a podcast content between two people ...
8,Generate a podcast content between two people ...
9,Generate a podcast content between two people ...


### Base model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import tqdm

tokenizer_1B = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
model_1B = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")

base_model_output = []
for q in tqdm.tqdm(query):
    input_text = q
    inputs = tokenizer_1B(input_text, return_tensors="pt")
    outputs = model_1B.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        eos_token_id=tokenizer_1B.eos_token_id,
    )
    generated_text = tokenizer_1B.decode(outputs[0], skip_special_tokens=True)
    base_model_output.append(generated_text)

In [ ]:
df = pd.DataFrame({
    'query': query,
    'base_model_output': base_model_output,
})
df.to_csv('evaluation_results_base.csv', index=False)

In [7]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

# 從 Hugging Face 載入模型
config = PeftConfig.from_pretrained("coconut19/llama-dialog-lora")
base_model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path)
model = PeftModel.from_pretrained(base_model, "coconut19/llama-dialog-lora")
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


adapter_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/13.6M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [10]:
import torch
import tqdm
model.eval()

def generate_dialog(user_instruction, max_new_tokens=512):
    prompt = f"Instruction: {user_instruction}\nAnswer:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_length=50,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text[len(prompt):].strip()

lora_model_output = []
for q in tqdm.tqdm(query):
    generated_text = generate_dialog(q, max_new_tokens=100)
    lora_model_output.append(generated_text)

100%|██████████| 50/50 [36:11<00:00, 43.42s/it]


In [11]:
df = pd.DataFrame({
    'query': query,
    'lora_model_output': lora_model_output
})
df.to_csv('evaluation_results_lora.csv', index=False)